# Calculate NDWI for Sentinel-2 Image

This notebook calculates the Normalized Difference Water Index (NDWI) for a Sentinel-2 image.

## What is NDWI?

NDWI (Normalized Difference Water Index) is a water index that uses the difference between green and near-infrared (NIR) light reflectance to detect water bodies and assess moisture content. Values range from -1 to 1:

- **Values close to 1**: Water bodies (open water, lakes, rivers)
- **Values around 0**: Bare soil or sparse vegetation
- **Values close to -1**: Dense vegetation or dry surfaces

## Parameters

This notebook has been automatically configured with the following parameters:

- **Collection**: {{STAC_COLLECTION_NAME}}
- **STAC Item**: {{STAC_ITEM_LINK}}
- **AOI (Area of Interest)**: {{AOI}} *(optional - if not provided, a 1000x1000 pixel sample will be used)*

## Workflow

1. Load the STAC item
2. Access the green (B03) and NIR (B08) bands
3. Calculate NDWI using the formula: (Green - NIR) / (Green + NIR)
4. Visualize the results

## Step 1: Import Required Libraries

In [ ]:
import pystac
import rasterio
from rasterio.windows import Window
from rasterio.mask import mask
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import json
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

## Step 2: Determine Area of Interest (AOI)

Determine the area to process. If an AOI is provided, clip to that geometry. Otherwise, extract a 1000x1000 pixel sample from the center of the image.

In [ ]:
# AOI parameter - must be GeoJSON geometry (Polygon, MultiPolygon, etc.)
# Using triple quotes to safely handle JSON strings with quotes
aoi_param = """{{AOI}}""".strip()

# Default window size if no AOI provided
DEFAULT_WINDOW_SIZE = 1000  # pixels

aoi_geometry = None
clip_window = None
use_windowed_read = False

# Check if AOI is provided
if aoi_param and aoi_param.lower() not in ["", "none", "null"]:
    try:
        # Parse as JSON
        aoi_data = json.loads(aoi_param)

        # Extract geometry from GeoJSON structure
        if aoi_data.get("type") == "FeatureCollection":
            # Extract first geometry from FeatureCollection
            if aoi_data.get("features") and len(aoi_data["features"]) > 0:
                aoi_geometry = aoi_data["features"][0].get("geometry")
        elif aoi_data.get("type") == "Feature":
            # Extract geometry from Feature
            aoi_geometry = aoi_data.get("geometry")
        elif aoi_data.get("type") in ["Polygon", "MultiPolygon", "Point", "LineString"]:
            # Direct geometry object
            aoi_geometry = aoi_data
        else:
            raise ValueError(f"Unsupported GeoJSON type: {aoi_data.get('type')}")

        # Validate geometry was extracted
        if aoi_geometry and aoi_geometry.get("type"):
            print(f"AOI provided: {aoi_geometry['type']} geometry")
            print("Will clip raster to AOI geometry")
        else:
            raise ValueError("Could not extract geometry from GeoJSON")

    except (json.JSONDecodeError, ValueError, KeyError) as e:
        print(f"Warning: Could not parse AOI as GeoJSON: {e}")
        print("Falling back to default 1000x1000 pixel window")
        aoi_geometry = None
else:
    print("No AOI provided, using default 1000x1000 pixel window")

# Set up windowed read if no AOI geometry
if aoi_geometry is None:
    use_windowed_read = True
    print(
        f"Will extract {DEFAULT_WINDOW_SIZE}x{DEFAULT_WINDOW_SIZE} pixel window from center"
    )

In [ ]:
# STAC item URL - automatically populated from selected dataset
stac_item_url = "{{STAC_ITEM_LINK}}"
stac_collection_name = "{{STAC_COLLECTION_NAME}}"

try:
    # Load the STAC item
    item = pystac.Item.from_file(stac_item_url)

    print(f"Successfully loaded STAC item: {item.id}")
    print(f"Collection: {stac_collection_name}")
    print(f"Date: {item.datetime}")
    print(f"Geometry: {item.geometry}")

except Exception as e:
    print(f"Error loading STAC item: {e}")
    raise

## Step 3: Access Sentinel-2 Bands

For NDWI calculation, we need:
- **Green band (B03)**: Wavelength ~560 nm
- **NIR band (B08)**: Wavelength ~842 nm

In [ ]:
try:
    # Find the green band (B03) and NIR band (B08) - either as separate assets or inside a multi-band COG
    green_band_asset = None
    nir_band_asset = None
    cog_asset = None  # Multi-band COG containing all bands (e.g. CEDA Sentinel-2 ARD)
    green_band_index = 3  # 1-based band index for B03 in standard S2 order
    nir_band_index = 8  # 1-based band index for B08 in standard S2 order

    # 1) Look for separate per-band assets first
    for asset_key, asset in item.assets.items():
        if "B03" in asset_key.upper() or (
            "green" in asset_key.lower() and "visual" not in asset_key.lower()
        ):
            green_band_asset = asset
            print(f"Found green band (B03): {asset_key}")
        if "B08" in asset_key.upper() or (
            "nir" in asset_key.lower() and "B08" in asset_key.upper()
        ):
            nir_band_asset = asset
            print(f"Found NIR band (B08): {asset_key}")

    # 2) If no per-band assets, look for a single multi-band COG (e.g. 'cog', 'data', 'image')
    if green_band_asset is None or nir_band_asset is None:
        for asset_key in ["cog", "data", "image", "reflectance", "bands"]:
            if asset_key in item.assets:
                cog_asset = item.assets[asset_key]
                # Try to get band indices from STAC eo:bands if present
                extra = getattr(cog_asset, "extra", {}) or {}
                eo_bands = extra.get("eo:bands", item.properties.get("eo:bands", []))
                if eo_bands:
                    for i, b in enumerate(eo_bands):
                        name = (b.get("name") or b.get("common_name") or "").upper()
                        if "B03" in name or b.get("common_name") == "green":
                            green_band_index = i + 1
                        if "B08" in name or b.get("common_name") == "nir":
                            nir_band_index = i + 1
                print(
                    f"Using multi-band asset '{asset_key}' (B03=band {green_band_index}, B08=band {nir_band_index})"
                )
                break
        if cog_asset is None:
            raise ValueError(
                "Could not find green (B03) or NIR (B08) bands. "
                "This STAC item has no per-band assets and no multi-band 'cog'/'data' asset."
            )

    if green_band_asset is not None and nir_band_asset is not None:
        print("\nBand assets found successfully!")
        print(f"Green band href: {green_band_asset.href}")
        print(f"NIR band href: {nir_band_asset.href}")
    else:
        print(f"\nMulti-band COG href: {cog_asset.href}")

except Exception as e:
    print(f"Error accessing bands: {e}")
    print("\nAvailable assets:")
    for asset_key in item.assets.keys():
        print(f"  - {asset_key}")
    raise

## Step 4: Read Band Data

Read the green and NIR band data into numpy arrays. If an AOI was provided, the data will be clipped to that geometry. Otherwise, a 1000x1000 pixel window will be extracted from the center.

In [ ]:
try:
    # Ensure AOI variables are initialized (in case AOI cell was not executed)
    try:
        _ = aoi_geometry
    except NameError:
        aoi_geometry = None
        clip_window = None
        use_windowed_read = False

    if cog_asset is not None:
        # Read B03 and B08 from multi-band COG (1-based band indices)
        with rasterio.open(cog_asset.href) as src:
            if src.count < max(green_band_index, nir_band_index):
                raise ValueError(
                    f"COG has {src.count} bands; need at least band {max(green_band_index, nir_band_index)}. "
                    "Check band order (e.g. B01,B02,B03,B04,...)."
                )

            # Determine clip window or use default window
            if aoi_geometry is not None:
                # Clip to AOI geometry
                green_data, green_transform = mask(
                    src, [aoi_geometry], crop=True, indexes=[green_band_index]
                )
                nir_data, nir_transform = mask(
                    src, [aoi_geometry], crop=True, indexes=[nir_band_index]
                )
                green_data = green_data[0]  # Remove band dimension
                nir_data = nir_data[0]
                # Update profile with clipped bounds
                green_profile = src.profile.copy()
                green_profile.update(
                    {
                        "height": green_data.shape[0],
                        "width": green_data.shape[1],
                        "transform": green_transform,
                        "count": 1,
                    }
                )
                print(f"Clipped to AOI geometry: {green_data.shape}")
            elif use_windowed_read:
                # Extract center 1000x1000 pixel window
                height, width = src.height, src.width
                if height < DEFAULT_WINDOW_SIZE or width < DEFAULT_WINDOW_SIZE:
                    print(
                        f"Image is smaller than {DEFAULT_WINDOW_SIZE}x{DEFAULT_WINDOW_SIZE}, using full image"
                    )
                    clip_window = None
                else:
                    row_off = (height - DEFAULT_WINDOW_SIZE) // 2
                    col_off = (width - DEFAULT_WINDOW_SIZE) // 2
                    clip_window = Window(
                        col_off, row_off, DEFAULT_WINDOW_SIZE, DEFAULT_WINDOW_SIZE
                    )
                    print(
                        f"Extracting {DEFAULT_WINDOW_SIZE}x{DEFAULT_WINDOW_SIZE} pixel window from center"
                    )

                green_data = src.read(green_band_index, window=clip_window)
                nir_data = src.read(nir_band_index, window=clip_window)
                green_profile = src.profile.copy()
                if clip_window:
                    green_profile.update(
                        {
                            "height": clip_window.height,
                            "width": clip_window.width,
                            "transform": rasterio.windows.transform(
                                clip_window, src.transform
                            ),
                            "count": 1,
                        }
                    )
                else:
                    green_profile.update(count=1)
            else:
                # Read full image
                green_data = src.read(green_band_index)
                nir_data = src.read(nir_band_index)
                green_profile = src.profile.copy()
                green_profile.update(count=1)

            green_crs = src.crs
            print(
                f"Read from multi-band COG: band {green_band_index} (green), band {nir_band_index} (NIR)"
            )
    else:
        # Read from separate band assets
        with rasterio.open(green_band_asset.href) as green_src:
            if aoi_geometry is not None:
                green_data, green_transform = mask(green_src, [aoi_geometry], crop=True)
                green_data = green_data[0]
                green_profile = green_src.profile.copy()
                green_profile.update(
                    {
                        "height": green_data.shape[0],
                        "width": green_data.shape[1],
                        "transform": green_transform,
                        "count": 1,
                    }
                )
            elif use_windowed_read:
                height, width = green_src.height, green_src.width
                if height < DEFAULT_WINDOW_SIZE or width < DEFAULT_WINDOW_SIZE:
                    clip_window = None
                else:
                    row_off = (height - DEFAULT_WINDOW_SIZE) // 2
                    col_off = (width - DEFAULT_WINDOW_SIZE) // 2
                    clip_window = Window(
                        col_off, row_off, DEFAULT_WINDOW_SIZE, DEFAULT_WINDOW_SIZE
                    )
                green_data = green_src.read(1, window=clip_window)
                green_profile = green_src.profile.copy()
                if clip_window:
                    green_profile.update(
                        {
                            "height": clip_window.height,
                            "width": clip_window.width,
                            "transform": rasterio.windows.transform(
                                clip_window, green_src.transform
                            ),
                        }
                    )
            else:
                green_data = green_src.read(1)
                green_profile = green_src.profile.copy()
            green_crs = green_src.crs

        with rasterio.open(nir_band_asset.href) as nir_src:
            if aoi_geometry is not None:
                nir_data, nir_transform = mask(nir_src, [aoi_geometry], crop=True)
                nir_data = nir_data[0]
            elif use_windowed_read:
                nir_data = nir_src.read(1, window=clip_window)
            else:
                nir_data = nir_src.read(1)
            # Use same profile as green (same grid)
            green_profile = nir_src.profile.copy()
            if aoi_geometry is not None:
                green_profile.update(
                    {
                        "height": nir_data.shape[0],
                        "width": nir_data.shape[1],
                        "transform": nir_transform,
                        "count": 1,
                    }
                )
            elif clip_window:
                green_profile.update(
                    {
                        "height": clip_window.height,
                        "width": clip_window.width,
                        "transform": rasterio.windows.transform(
                            clip_window, nir_src.transform
                        ),
                    }
                )

    print(f"Green band shape: {green_data.shape}, dtype: {green_data.dtype}")
    print(f"NIR band shape: {nir_data.shape}, dtype: {nir_data.dtype}")
    print(f"CRS: {green_crs}")

    if green_data.shape != nir_data.shape:
        raise ValueError(
            f"Band shapes do not match: Green {green_data.shape} vs NIR {nir_data.shape}"
        )

    green_data = green_data.astype(np.float32)
    nir_data = nir_data.astype(np.float32)
    print(f"\nBand data loaded successfully! Processing {green_data.size:,} pixels")

except Exception as e:
    print(f"Error reading band data: {e}")
    raise

## Step 5: Calculate NDWI

Calculate NDWI using the formula:

$$NDWI = \frac{Green - NIR}{Green + NIR}$$

The result will be a value between -1 and 1.

In [ ]:
try:
    # Calculate NDWI
    # Avoid division by zero by adding a small epsilon
    denominator = green_data + nir_data

    # Create mask for valid pixels (where denominator is not zero)
    valid_mask = denominator != 0

    # Initialize NDWI array with NaN for invalid pixels
    ndwi = np.full_like(green_data, np.nan, dtype=np.float32)

    # Calculate NDWI only for valid pixels
    ndwi[valid_mask] = (green_data[valid_mask] - nir_data[valid_mask]) / denominator[
        valid_mask
    ]

    # Clip values to valid NDWI range [-1, 1]
    ndwi = np.clip(ndwi, -1.0, 1.0)

    print("NDWI calculation complete!")
    print(f"NDWI shape: {ndwi.shape}")
    print(f"NDWI min: {np.nanmin(ndwi):.4f}")
    print(f"NDWI max: {np.nanmax(ndwi):.4f}")
    print(f"NDWI mean: {np.nanmean(ndwi):.4f}")
    print(f"Valid pixels: {np.sum(~np.isnan(ndwi)):,} out of {ndwi.size:,}")

except Exception as e:
    print(f"Error calculating NDWI: {e}")
    raise

## Step 6: Visualize Results

Create visualizations of the NDWI results.

In [ ]:
# Create a custom colormap for NDWI visualization
# Colors: dense vegetation (brown) -> sparse vegetation (yellow) -> bare soil (gray) -> water (blue)
colors = ["#8B4513", "#D2691E", "#CCCCCC", "#87CEEB", "#4169E1", "#000080"]
n_bins = 256
cmap = LinearSegmentedColormap.from_list("ndwi", colors, N=n_bins)

# Create figure with subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Create visualization of green band
im1 = axes[0].imshow(
    green_data, cmap="Greens", vmin=0, vmax=np.percentile(green_data, 98)
)
axes[0].set_title("Green Band (B03)", fontsize=14, fontweight="bold")
axes[0].axis("off")
plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04, label="Reflectance")

# Create visualization of NIR band
im2 = axes[1].imshow(nir_data, cmap="YlGn", vmin=0, vmax=np.percentile(nir_data, 98))
axes[1].set_title("NIR Band (B08)", fontsize=14, fontweight="bold")
axes[1].axis("off")
plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04, label="Reflectance")

# Create visualization of NDWI
im3 = axes[2].imshow(ndwi, cmap=cmap, vmin=-1, vmax=1)
axes[2].set_title("NDWI", fontsize=14, fontweight="bold")
axes[2].axis("off")
cbar = plt.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04, label="NDWI")

# Add colorbar labels
cbar.set_ticks([-1, -0.5, 0, 0.3, 0.6, 1])
cbar.set_ticklabels(
    [
        "Dense Veg",
        "Sparse Veg",
        "Bare Soil",
        "Moist Soil",
        "Shallow Water",
        "Deep Water",
    ]
)

plt.suptitle(f"NDWI Analysis - {item.id}", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("Visualization complete!")

## Summary

This notebook has successfully:

1. ✅ Loaded the STAC item from: `{{STAC_ITEM_LINK}}`
2. ✅ Determined area of interest (AOI or default window)
3. ✅ Accessed the green (B03) and NIR (B08) bands
4. ✅ Calculated NDWI values
5. ✅ Visualized the results

### Next Steps

You can now:
- Analyze specific areas of interest
- Export the NDWI results for further analysis
- Compare NDWI values across different dates
- Create time series analyses for water body monitoring

### Resources

- **Collection**: {{STAC_COLLECTION_NAME}}

---

*This notebook was automatically generated from a template and configured with your selected dataset.*